# News Topic Classifier — BERT fine-tuning on AG News

Run this notebook top-to-bottom on **Google Colab** (Runtime > Change runtime type > GPU) for a free GPU.
It reproduces the exact pipeline in the `news-topic-classifier` project: data prep, fine-tuning, evaluation, and a quick inference demo.

**Fixed version notes:**
- Package versions are now pinned to a tested, mutually-compatible set (this avoids the `ImportError: cannot import name 'kernelize'` and similar version-mismatch errors).
- After the install cell runs for the very first time in a fresh runtime, you do **not** need to restart — pinned installs avoid the upgrade-mid-session problem. If Colab still shows a "RESTART RUNTIME" button after the install cell, click it once, then just continue from the next cell.
- Added a GPU check so `fp16` is only enabled when a GPU is actually available (otherwise training crashes).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_PATH = "/content/drive/MyDrive/bert-ag-news-model"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print("Model will be saved to:", DRIVE_SAVE_PATH)

Mounted at /content/drive
Model will be saved to: /content/drive/MyDrive/bert-ag-news-model


In [4]:
import os
print(os.listdir(DRIVE_SAVE_PATH))

[]


In [5]:
!pip install -q \
  "transformers==4.46.3" \
  "datasets==2.21.0" \
  "accelerate==0.34.2" \
  "evaluate==0.4.3" \
  "scikit-learn==1.5.2" \
  "huggingface_hub==0.25.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incomp

In [6]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

SEED = 42
set_seed(SEED)

MODEL_CHECKPOINT = "bert-base-uncased"
MAX_LEN = 128
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in enumerate(LABEL_NAMES)}

USE_GPU = torch.cuda.is_available()
print("CUDA available:", USE_GPU)
if not USE_GPU:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU, then Runtime > Restart session.")

CUDA available: True


## 1. Load & tokenize AG News

In [7]:
raw = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

tokenized = raw.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])
tokenized

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

## 2. Fine-tune bert-base-uncased

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

args = TrainingArguments(
    output_dir="bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    fp16=USE_GPU,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.176200,0.184756,0.942632,0.942692,0.942692
2,0.131700,0.187001,0.947368,0.947388,0.947388
3,0.097500,0.226317,0.948026,0.948033,0.948033


TrainOutput(global_step=22500, training_loss=0.17332125778198243, metrics={'train_runtime': 3111.8453, 'train_samples_per_second': 115.687, 'train_steps_per_second': 7.23, 'total_flos': 2.368042020864e+16, 'train_loss': 0.17332125778198243, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Evaluate

In [9]:
preds_output = trainer.predict(tokenized["test"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("F1 macro:", f1_score(labels, preds, average="macro"))
print(classification_report(labels, preds, target_names=LABEL_NAMES))
print(confusion_matrix(labels, preds))

Accuracy: 0.9480263157894737
F1 macro: 0.9480332708780671
              precision    recall  f1-score   support

       World       0.96      0.96      0.96      1900
      Sports       0.99      0.99      0.99      1900
    Business       0.93      0.91      0.92      1900
    Sci/Tech       0.91      0.94      0.92      1900

    accuracy                           0.95      7600
   macro avg       0.95      0.95      0.95      7600
weighted avg       0.95      0.95      0.95      7600

[[1824    9   33   34]
 [  11 1873    9    7]
 [  35    7 1727  131]
 [  25    9   85 1781]]


## 4. Save model (download this folder to use in the Streamlit/Gradio app)

In [10]:
import os

os.makedirs("models/bert-ag-news", exist_ok=True)
trainer.save_model("models/bert-ag-news")
tokenizer.save_pretrained("models/bert-ag-news")

# Optional: zip it for easy download from Colab
!zip -r bert-ag-news-model.zip models/bert-ag-news

  adding: models/bert-ag-news/ (stored 0%)
  adding: models/bert-ag-news/tokenizer_config.json (deflated 76%)
  adding: models/bert-ag-news/model.safetensors (deflated 7%)
  adding: models/bert-ag-news/tokenizer.json (deflated 71%)
  adding: models/bert-ag-news/special_tokens_map.json (deflated 42%)
  adding: models/bert-ag-news/training_args.bin (deflated 53%)
  adding: models/bert-ag-news/config.json (deflated 51%)
  adding: models/bert-ag-news/vocab.txt (deflated 53%)


## 5. Quick inference test

In [12]:
import torch
import torch.nn.functional as F

sample_texts = [
    "Apple unveils new AI chip that boosts iPhone performance",
    "Local team wins championship after dramatic overtime finish",
    "Central bank raises interest rates amid inflation concerns",
    "NASA's new telescope captures stunning images of a distant galaxy",
]

model.eval()
for text in sample_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1).squeeze()
    pred = ID2LABEL[int(probs.argmax())]
    print(f"{text}\n -> Predicted: {pred} (confidence {probs.max():.2f})\n")

Apple unveils new AI chip that boosts iPhone performance
 -> Predicted: Sci/Tech (confidence 0.99)

Local team wins championship after dramatic overtime finish
 -> Predicted: Sports (confidence 0.98)

Central bank raises interest rates amid inflation concerns
 -> Predicted: Business (confidence 0.97)

NASA's new telescope captures stunning images of a distant galaxy
 -> Predicted: Sci/Tech (confidence 0.95)

